## Изучение тюнинг-эффекта с помощью 1D FD-моделирования: анализ зависимости амплитуды отражений от мощности пласта.

Tuning эффект (эффект настройки, эффект резонанса) — это фундаментальное явление в сейсморазведке, при котором амплитуда сейсмического отражения от тонкого слоя максимально усиливается при определенном соотношении между мощностью слоя и длиной волны сейсмического сигнала.

Физическая суть явления

<b>1. Интерференция волн как основной механизм</b>

Когда сейсмическая волна падает на тонкий слой, возникают два отражения:

* От верхней границы слоя (R₁)
* От нижней границы слоя (R₂)
Эти две волны интерферируют (складываются) при возвращении к приемнику. Tuning эффект возникает, когда:

Разность хода между волнами создает конструктивную интерференцию
Фазовые сдвиги при отражении усиливают этот эффект

<b>2. Условие максимального усиления</b>

Максимум амплитуды достигается при мощности слоя ```h = λ/4```, где:

* ```λ``` — длина волны в материале слоя
* ```λ = V / f```, где ```V``` — скорость в слое, ```f``` — доминирующая частота сигнала
Почему именно λ/4?

* Волна, отразившаяся от нижней границы, проходит путь ```2h = 2 × (λ/4) = λ/2```
* Путь ```λ/2``` означает разность фаз 180°
* При отражении от нижней границы (если это переход от низкого импеданса к высокому) происходит дополнительный сдвиг фазы на 180°
* Суммарный сдвиг: ```180° + 180° = 360° = 0°``` → полное конструктивное сложение

### План работы:
* Создание одномерных моделей с разной мощностью слоя 
* Получение соответствующих трасс с помощью 1D моделирования и сравнение на общем wiggle plot
* Построение графика зависимости амплитуды отражения от мощности слоя


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from tqdm.notebook import tqdm

### Функции для 1D FD моделирования

In [ ]:
def update_d2pz_9pt(p, dz, nz, d2pz):
    for j in range(4, nz-4):
        d2pz[j] = (-p[j+4]/560+8*p[j+3]/315-p[j+2]/5+8*p[j+1]/5-205*p[j]/72+8*p[j-1]/5-p[j-2]/5+8*p[j-3]/315-p[j-4]/560)/(dz**2)
    return d2pz

def absorb_1d(nz, thickness):
    FW = thickness
    a = 0.0053
    
    coeff = np.zeros(FW).astype('float32')
    for i in range(FW):
        coeff[i] = np.exp(-(a**2 * (FW-i)**2))

    absorb_coeff = np.ones(nz).astype('float32')

    for j in range(FW):
        absorb_coeff[j] = coeff[j]

    for j in range(FW):
        jj = nz - j - 1
        absorb_coeff[jj] = coeff[j]

    return absorb_coeff

def fd1d_forward(src, vp, nt, dt, dz, zsrc, n_absorb):
    nz_orig = vp.shape[0]
    
    vp_padded = np.pad(vp, n_absorb, mode='edge')
    nz = vp_padded.shape[0]
    
    jsrc = int(zsrc/dz) + n_absorb

    absorb_coeff = absorb_1d(nz, n_absorb)
   
    vp2 = vp_padded**2
    
    # Всегда сохраняем все шаги подряд
    u = np.zeros((3, nz)).astype('float32')
    u_snapshots = np.zeros((nt, nz)).astype('float32')
    
    d2pz = np.zeros(nz).astype('float32')
    
    # Сохраняем начальные условия (it=0 и it=1)
    u_snapshots[0] = u[0]
    if nt > 1:
        u_snapshots[1] = u[0]  # it=1 тоже начальное условие
    
    for it in tqdm(range(2, nt), desc='Time iterations'):
        idx_prev2 = (it - 2) % 3
        idx_prev1 = (it - 1) % 3
        idx_curr = it % 3
    
        d2pz = update_d2pz_9pt(u[idx_prev1], dz, nz, d2pz)
           
        u[idx_curr] = 2 * u[idx_prev1] - u[idx_prev2] + vp2 * dt**2 * d2pz
   
        u[idx_curr, jsrc] = u[idx_curr, jsrc] + src[it] / dz * dt ** 2
        u[idx_prev1] *= absorb_coeff
        u[idx_curr] *= absorb_coeff
        
        # Сохраняем каждый шаг
        u_snapshots[it] = u[idx_curr].copy()
    
    u_cropped = u_snapshots[:, n_absorb:nz-n_absorb]
       
    return u_cropped


### Функция создания 1D модели с заданным размером и параметрами слоя

In [ ]:
def build_model(height, dz, layer_top, layer_height, v_background, v_layer):
    nz = int(height / dz)
    vp = np.ones(nz) * v_background
    
    layer_top_idx = int(layer_top / dz)
    layer_bottom_idx = int((layer_top + layer_height) / dz)
    
    layer_top_idx = max(0, min(layer_top_idx, nz - 1))
    layer_bottom_idx = max(0, min(layer_bottom_idx, nz))
    
    vp[layer_top_idx:layer_bottom_idx] = v_layer
    
    return vp

def ricker_wavelet(freq, dt, nt):
    t = np.arange(nt) * dt
    t0 = 1.0 / freq
    a = np.pi * freq * (t - t0)
    src = (1 - 2 * a**2) * np.exp(-a**2)
    src = src * 1e6
    return src

def get_trace(vel_model, dz, total_time, src_freq, zsrc, zrec):
    nz = vel_model.shape[0]
    
    v_max = np.max(vel_model)
    C = 0.5
    dt = C * dz / v_max
    
    # Вычисляем nt так, чтобы максимальное время было >= total_time
    # nt должен быть таким, что (nt-1) * dt >= total_time
    nt = int(np.ceil(total_time / dt)) + 1
    
    jrec = int(zrec / dz)
    
    src = ricker_wavelet(src_freq, dt, nt)
    
    u = fd1d_forward(src, vel_model, nt, dt, dz, zsrc, n_absorb=300)
    
    trace = u[:, jrec]
    
    return trace, dt, src, u


### Задание параметров 1D-моделей

In [ ]:
# Параметры модели
height = 400
dz = 1
layer_top = 250
v_background = 2000
v_layer = 2500

# Параметры моделирования
src_freq = 30  # фиксированная частота (Гц)
total_time = 0.4
order = 9
min_depth = (order // 2) * dz
zsrc = min_depth + 5 * dz
zrec = min_depth + 5 * dz  # zero-offset

# Вычисляем длину волны для тюнинг эффекта
# lambda = v / f, где v - скорость в слое, f - частота
wavelength = v_layer / src_freq
print(f'Длина волны в слое: {wavelength:.2f} м')

# Рассчитываем ряд мощностей для теста
# Через 2м до lambda/4 и далее через 5м до lambda
thicknesses = list(range(2, 22, 2)) + list(range(23, 85, 5))

# Округляем до шага сетки
thicknesses = [round(t / dz) * dz for t in thicknesses]
thicknesses = sorted(set(thicknesses))  # убираем дубликаты и сортируем

print(f'Мощности для теста: {thicknesses}')


Стыковка 1D-моделей в двумерную для наглядности

In [ ]:
# Построение глубинно-скоростной модели (все модели совмещены в 2D)
nz = int(height / dz)
depths = np.arange(nz) * dz

# Создаем 2D массив: каждая колонка - это одна модель скорости
vel_models_2d = np.zeros((nz, len(thicknesses)))

for idx, layer_height in enumerate(thicknesses):
    vel_model = build_model(height, dz, layer_top, layer_height, v_background, v_layer)
    vel_models_2d[:, idx] = vel_model

# Визуализация
fig, ax = plt.subplots(figsize=(10, 6))

# Создаем изображение
im = ax.imshow(vel_models_2d, aspect='auto', origin='upper', 
               extent=[thicknesses[0], thicknesses[-1], depths[-1], depths[0]],
               cmap='PuOr', vmin=v_background, vmax=v_layer)

# Настройка осей
ax.set_xlabel('Мощность слоя (м)', fontsize=12)
ax.set_ylabel('Глубина (м)', fontsize=12)
ax.set_title(f'Глубинно-скоростная модель для разных мощностей слоя\n(v_background={v_background} м/с, v_layer={v_layer} м/с)', fontsize=14)

# Настройка меток по оси X (мощности слоя)
# Показываем не все метки, чтобы не перегружать график
step = max(1, len(thicknesses) // 10)
x_ticks = thicknesses[::step]
ax.set_xticks(x_ticks)
ax.set_xticklabels([f'{h:.0f}' for h in x_ticks], rotation=45, ha='right')

# Добавляем цветовую шкалу
cbar = plt.colorbar(im, ax=ax)
cbar.set_label('Скорость (м/с)', fontsize=12)

# Добавляем метку для lambda/4
lambda_4 = wavelength / 4
lambda_4_idx = None
min_diff = float('inf')
for idx, layer_height in enumerate(thicknesses):
    diff = abs(layer_height - lambda_4)
    if diff < min_diff:
        min_diff = diff
        lambda_4_idx = idx

if lambda_4_idx is not None:
    lambda_4_thickness = thicknesses[lambda_4_idx]
    ax.axvline(x=lambda_4_thickness, color='yellow', linestyle='--', linewidth=2, alpha=0.8)
    ax.text(lambda_4_thickness, depths[0] + 20, 'λ/4', ha='center', va='bottom', 
            color='yellow', fontsize=12, fontweight='bold',
            bbox=dict(boxstyle='round,pad=0.3', facecolor='black', alpha=0.7, edgecolor='yellow'))

plt.tight_layout()
plt.show()


### 1D-моделирование для каждой модели

In [ ]:
# Перебор мощностей и сохранение трасс
traces_dict = {}

for layer_height in tqdm(thicknesses, desc='Перебор мощностей'):
    # Создаем модель с текущей мощностью слоя
    vel_model = build_model(height, dz, layer_top, layer_height, v_background, v_layer)
    
    # Получаем трассу
    trace, dt, src, u = get_trace(vel_model, dz, total_time, src_freq, zsrc, zrec)
    
    # Сохраняем в словарь
    traces_dict[layer_height] = {
        'trace': trace,
        'dt': dt,       
        'wavelength': wavelength,
        'layer_height': layer_height
    }

print(f'Сохранено трасс: {len(traces_dict)}')


Построение wiggle plot для сравнения

In [ ]:
# Визуализация всех трасс (wiggle plot)
fig, ax = plt.subplots(figsize=(12, 5))

# Параметры для отсечения начальных 0.15 сек
cut_time = 0.2
trace_offset = 1.0  # смещение между трассами

sorted_items = sorted(traces_dict.items())

# Предвычисляем данные для всех трасс (избегаем дублирования вычислений)
precomputed_data = []
global_max_amp = 0.0

for layer_height, data in sorted_items:
    trace = data['trace']
    dt = data['dt']
    time_axis = np.arange(len(trace)) * dt
    
    # Отсекаем начальные 0.2 сек
    cut_idx = int(cut_time / dt)
    if cut_idx < len(trace):
        trace_cut = trace[cut_idx:]
        time_cut = time_axis[cut_idx:]
    else:
        trace_cut = trace
        time_cut = time_axis
    
    # Сохраняем предвычисленные значения
    precomputed_data.append({
        'layer_height': layer_height,
        'trace_cut': trace_cut,
        'time_cut': time_cut
    })
    
    # Находим общий максимум
    max_amp = np.max(np.abs(trace_cut))
    if max_amp > global_max_amp:
        global_max_amp = max_amp

# Теперь рисуем все трассы в едином масштабе
for idx, precomp in enumerate(precomputed_data):
    trace_cut = precomp['trace_cut']
    time_cut = precomp['time_cut']
    
    # Нормализация трассы по общему максимуму
    if global_max_amp > 0:
        trace_normalized = trace_cut / global_max_amp * 0.8  # масштабирование
    else:
        trace_normalized = trace_cut
    
    # Смещение трассы по X
    x_offset = idx * trace_offset
    
    # Рисуем трассу (время по Y, увеличивается вниз)
    ax.plot(x_offset + trace_normalized, time_cut, 'k-', linewidth=0.5)
    
    # Закрашиваем положительные значения
    positive_mask = trace_normalized > 0
    if np.any(positive_mask):
        ax.fill_betweenx(time_cut[positive_mask], 
                         x_offset, 
                         x_offset + trace_normalized[positive_mask],
                         color='black', alpha=0.6)

# Настройка ticks и ticklabels сверху
x_positions = [idx * trace_offset for idx in range(len(precomputed_data))]
thickness_labels = [f'{precomp["layer_height"]:.0f}' for precomp in precomputed_data]

ax.set_xticks(x_positions)
ax.set_xticklabels(thickness_labels)
ax.xaxis.set_ticks_position('top')
ax.xaxis.set_label_position('top')

# Добавляем метку для lambda/4
lambda_4 = wavelength / 4
# Находим ближайшую трассу к lambda/4
lambda_4_idx = None
min_diff = float('inf')
for idx, precomp in enumerate(precomputed_data):
    diff = abs(precomp['layer_height'] - lambda_4)
    if diff < min_diff:
        min_diff = diff
        lambda_4_idx = idx

# Вычисляем x_lambda_4 один раз
x_lambda_4 = None
if lambda_4_idx is not None:
    x_lambda_4 = lambda_4_idx * trace_offset
    ax.axvline(x=x_lambda_4, color='red', linestyle='--', linewidth=1.5, alpha=0.7, label='λ/4')

# Добавляем метку для lambda/2
lambda_2 = wavelength / 2
# Находим ближайшую трассу к lambda/2
lambda_2_idx = None
min_diff = float('inf')
for idx, precomp in enumerate(precomputed_data):
    diff = abs(precomp['layer_height'] - lambda_2)
    if diff < min_diff:
        min_diff = diff
        lambda_2_idx = idx

# Вычисляем x_lambda_2 один раз
x_lambda_2 = None
if lambda_2_idx is not None:
    x_lambda_2 = lambda_2_idx * trace_offset
    ax.axvline(x=x_lambda_2, color='blue', linestyle='--', linewidth=1.5, alpha=0.7, label='λ/2')

ax.set_xlabel('Мощность слоя (м)')
ax.set_ylabel('Время (с)')
ax.set_title(f'Wiggle plot трасс для разных мощностей слоя (f={src_freq} Гц, λ={wavelength:.2f} м)')
ax.invert_yaxis()  # время увеличивается вниз

# Добавляем текстовую метку после invert_yaxis (чтобы знать правильные координаты)
if x_lambda_4 is not None:
    y_min, y_max = ax.get_ylim()
    # После invert_yaxis y_min - это верх (меньшее время), y_max - низ (большее время)
    ax.text(x_lambda_4, y_min, 'λ/4', ha='center', va='bottom', 
            color='red', fontsize=10, fontweight='bold', 
            bbox=dict(boxstyle='round,pad=0.3', facecolor='white', alpha=0.8, edgecolor='red'))

# Добавляем текстовую метку для lambda/2
if x_lambda_2 is not None:
    y_min, y_max = ax.get_ylim()
    # После invert_yaxis y_min - это верх (меньшее время), y_max - низ (большее время)
    ax.text(x_lambda_2, y_min, 'λ/2', ha='center', va='bottom', 
            color='blue', fontsize=10, fontweight='bold', 
            bbox=dict(boxstyle='round,pad=0.3', facecolor='white', alpha=0.8, edgecolor='blue'))
ax.grid(True, alpha=0.3, axis='y')
ax.set_xlim(-0.5, len(precomputed_data) * trace_offset - 0.5)
plt.tight_layout()
plt.show()


Можно заметить, что на трассе, соответствующей λ/4, амплитуда действительно максимальная (ниже на графике убедимся в этом).
И ещё, в теории на λ/2 должно происходить уверенное разделение границ. Да, действительно видно 2 раздельных пика

### Анализ амплитуды отражения

Для симметричного тонкого слоя амплитуда отражения должна возрастать до λ/4 по следующему закону:

* ```|R_total| = 2R |sin(2πh/λ)|```

Далее, на интервале ```λ/4 < h < λ/2``` будет переходная зона - затухание амплитуды по искаженной синусойде, 
а затем при разделении отражений амплитуда стабилизируется и будет константной.

Проверим?



In [ ]:
# Анализ амплитуд в промежутке 0.25-0.30 сек
time_min = 0.2
time_max = 0.4

amplitudes = {}
for layer_height, data in traces_dict.items():
    trace = data['trace']
    dt = data['dt']
    time_axis = np.arange(len(trace)) * dt
    
    # Выбираем индексы в нужном временном диапазоне
    mask = (time_axis >= time_min) & (time_axis <= time_max)
    
    if np.any(mask):
        trace_segment = trace[mask]
        max_amp = np.max(np.abs(trace_segment))
        amplitudes[layer_height] = max_amp
    else:
        amplitudes[layer_height] = 0.0

# Визуализация зависимости амплитуды от мощности
thicknesses_sorted = sorted(amplitudes.keys())
amps_sorted = [amplitudes[h] for h in thicknesses_sorted]
thicknesses_normalized = [h / wavelength for h in thicknesses_sorted]

# Вычисляем коэффициент отражения R
# Для нормального падения: R = (Z2 - Z1) / (Z2 + Z1), где Z = ρ * V
# Если плотности одинаковые: R ≈ (v2 - v1) / (v2 + v1)
# Для более точного расчета можно использовать формулу с импедансами
R = (v_layer - v_background) / (v_layer + v_background)

# Вычисляем теоретическую кривую
# Ограничиваем до lambda/4
lambda_4 = wavelength / 4
h_theory = np.linspace(min(thicknesses_sorted), lambda_4, 500)
h_theory_normalized = h_theory / wavelength
amplitude_theory = 2 * R * np.abs(np.sin(2 * np.pi * h_theory / wavelength))

# Нормализация обеих кривых для корректного сравнения
max_amp_simulation = max(amps_sorted) if amps_sorted else 1.0
max_amp_theory = np.max(amplitude_theory) if len(amplitude_theory) > 0 else 1.0

amps_sorted_normalized = [amp / max_amp_simulation for amp in amps_sorted] if max_amp_simulation > 0 else amps_sorted
amplitude_theory_normalized = amplitude_theory / max_amp_theory if max_amp_theory > 0 else amplitude_theory

fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(thicknesses_normalized, amps_sorted_normalized, 'o-', linewidth=2, markersize=8, label='Моделирование')
ax.plot(h_theory_normalized, amplitude_theory_normalized, 'r--', linewidth=2, alpha=0.7, label='Теоретическая кривая')
ax.axvline(x=0.25, color='r', linestyle='--', alpha=0.5)
ax.axvline(x=0.5, color='b', linestyle='--', alpha=0.5)
ax.axvline(x=1.0, color='g', linestyle='--', alpha=0.5)

# Добавляем текстовые подписи возле линий
y_max = max(amps_sorted_normalized)
y_min = min(amps_sorted_normalized)
y_range = y_max - y_min

# Подпись для λ/4
ax.text(0.25, y_max - 0.1 * y_range, 'λ/4', 
        ha='center', va='bottom', color='r', fontsize=12, fontweight='bold',
        bbox=dict(boxstyle='round,pad=0.3', facecolor='white', alpha=0.8, edgecolor='red'))

# Подпись для λ/2
ax.text(0.5, y_max - 0.08 * y_range, 'λ/2', 
        ha='center', va='bottom', color='b', fontsize=12, fontweight='bold',
        bbox=dict(boxstyle='round,pad=0.3', facecolor='white', alpha=0.8, edgecolor='blue'))

# Подпись для λ
ax.text(1.0, y_max - 0.05 * y_range, 'λ', 
        ha='center', va='bottom', color='g', fontsize=12, fontweight='bold',
        bbox=dict(boxstyle='round,pad=0.3', facecolor='white', alpha=0.8, edgecolor='green'))

ax.set_xlabel('Мощность слоя / Длина волны')
ax.set_ylabel(f'Нормализованная амплитуда (временной диапазон {time_min}-{time_max} с)')
ax.set_title(f'Зависимость амплитуды от мощности слоя (f={src_freq} Гц, R={R:.3f})')
ax.grid(True, alpha=0.3)
ax.legend()
plt.tight_layout()
plt.show()


Ну, почти :) Спишем несовпадения на неточности численной схемы. Главное, наблюдается чёткий пик на λ/4 и более-менее стабилизация после λ/2

Кстати, вот эту зависимость амплитуды от мощности в зоне < λ/4 можно использовать для оценки мощности пласта без фактической его отбивки по волновой картине!